# Qwen2-VL-7B-Instruct × Chitrakshara (QLoRA) → BharatBench evaluation vs. Chitrarth

This notebook (1) QLoRA-fine-tunes **Qwen2-VL-7B-Instruct** on **Chitrakshara-Cap**, (2) evaluates the base model **and** the fine-tuned model on **Indic-LLaVA-Bench** (GPT judge) and **Indic-POPE** (no API key needed) from this repo, and (3) puts the numbers next to **Chitrarth** and prints diagnostics for improving the model.

Runtime: Colab **A100** (40 or 80 GB). Colab secrets: `HF_TOKEN` (recommended), `OPENAI_API_KEY` (only for the LLaVA-Bench judge).

### What was broken in the previous version, and the fix

| Problem | Effect | Fix |
|---|---|---|
| Responses written without `text` (question), `category`, `model_name`, and with the dataset's own ids instead of BharatBench `question_id`s | `KeyError` in `answer_file.py`, so `review_Hindi` came out empty and **no Hindi score was produced** (the numbers printed were the repo's old Telugu sample) | The questions are read from `Indic-LLaVABench-Evaluator/inputs/<Lang>/questions.jsonl`, and responses are written in exactly the documented format |
| Only 50 of the 60 questions were answered | The judge pairs answers with questions **line by line** (`zip`), so the results were truncated/misaligned | All 60 are answered, in file order. The evaluator now refuses mismatched counts |
| `max_new_tokens=128` | Detailed answers got cut off, so the judge scored them low | 768 tokens |
| Only the fine-tuned model was scored | There was no way to tell whether fine-tuning helped | The **base model is scored with the same pipeline** (adapter disabled), so you get a real delta |
| Captions trained with the prompt *"Describe this image."* | Teaches the model to answer "describe" with a 10-word news headline, which **hurts** LLaVA-Bench (detail/complex questions) | Captions use explicit *short caption* prompts, and a self-distilled set of detailed Hindi answers is mixed in to keep instruction following |
| Training stopped at a hand-picked train loss (1.34) | Train loss on noisy alt-text is not a quality metric | Early stopping on **eval loss** + best checkpoint restore |
| Captions in every language and boilerplate prefixes (`Photo of …`) | Wasted capacity for a Hindi benchmark | Script filter (Devanagari for Hindi) and caption cleaning |
| Judge script: `openai==0.28` API, `gpt-4-0613` hard-coded, infinite retry loop, appends to stale review files | Hangs without a key, re-uses old reviews | Repo scripts fixed: openai>=1 client, `--judge-model`, bounded retries, per-model output files |

In [ ]:
!pip install -q -U bitsandbytes peft accelerate datasets transformers pillow requests tqdm matplotlib pandas "openai>=1.0" shortuuid

## 1. Config

In [ ]:
import os, random, math, json, re, gc
import numpy as np
import torch

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

CFG = dict(
    # Qwen/Qwen2-VL-7B-Instruct (as before). "Qwen/Qwen2.5-VL-7B-Instruct" also works and is a stronger base.
    model_id        = "Qwen/Qwen2-VL-7B-Instruct",
    dataset_id      = "krutrim-ai-labs/Chitrakshara",
    dataset_config  = "Chitrakshara-Cap",
    seed            = 42,

    # ---- evaluation target ----
    eval_language   = "Hindi",      # any of the 12 BharatBench languages
    run_name        = "qwen2vl_chitrakshara_qlora",

    # ---- data ----
    n_train         = 3000,         # caption pairs after filtering
    n_eval          = 120,
    candidate_pool  = 25000,        # raw rows to try (many URLs are dead / other languages)
    filter_script   = True,         # keep only captions written in the eval language's script
    min_script_frac = 0.6,
    max_caption_tok = 160,
    min_caption_chr = 15,
    max_caption_chr = 400,
    max_image_side  = 448,
    min_pixels      = 64  * 28 * 28,
    max_pixels      = 256 * 28 * 28,   # train-time visual tokens/image (memory saver)
    eval_max_pixels = 768 * 28 * 28,   # inference-time resolution (more detail for benchmarks)

    # ---- anti-forgetting: self-distilled detailed answers mixed into training ----
    n_distill       = 300,          # 0 to disable
    distill_max_tok = 384,

    # ---- (Q)LoRA ----
    use_4bit        = True,         # False -> plain bf16 LoRA (fits on 80 GB, usually a bit better)
    lora_r          = 16,
    lora_alpha      = 32,
    lora_dropout    = 0.05,

    # ---- optimisation ----
    lr              = 1e-4,
    epochs          = 1,
    per_device_bs   = 4,
    grad_accum      = 4,            # effective batch 16
    warmup_ratio    = 0.05,
    weight_decay    = 0.0,
    max_grad_norm   = 0.3,
    lr_scheduler    = "cosine",
    eval_steps      = 40,
    early_stop_patience = 3,

    # ---- generation ----
    gen_bs          = 8,
    llava_max_new_tokens = 768,
    pope_n_per_cat  = 300,          # None = full Indic-POPE (8,910 questions, what papers report)
    answer_lang_hint = False,       # True appends "answer in <language>" (changes the protocol; report separately)
)

OUT_DIR  = f"/content/{CFG['run_name']}"
SAVE_DIR = f"{OUT_DIR}/final_adapter"
BB_DIR   = "/content/BharatBench"
BB_BRANCH = "claude/gallant-goodall-i338or"   # branch with the fixed evaluator scripts; use "main" once merged

random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])
assert torch.cuda.is_available(), "Switch the Colab runtime to a GPU (A100 recommended)."
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
# Secrets (Colab: key icon on the left -> add HF_TOKEN / OPENAI_API_KEY)
try:
    from google.colab import userdata
    for k in ("HF_TOKEN", "OPENAI_API_KEY"):
        try:
            v = userdata.get(k)
            if v: os.environ[k] = v
        except Exception:
            pass
except ImportError:
    pass
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(os.environ["HF_TOKEN"])
print("HF_TOKEN:", "set" if os.environ.get("HF_TOKEN") else "missing",
      "| OPENAI_API_KEY:", "set" if os.environ.get("OPENAI_API_KEY") else "missing (LLaVA-Bench judge will be skipped)")

# BharatBench (evaluator scripts + question files)
if os.path.isdir(BB_DIR):
    !git -C {BB_DIR} fetch -q origin {BB_BRANCH} && git -C {BB_DIR} checkout -q {BB_BRANCH} && git -C {BB_DIR} pull -q origin {BB_BRANCH}
else:
    !git clone -q -b {BB_BRANCH} https://github.com/Koushik25022005/BharatBench.git {BB_DIR}
!git -C {BB_DIR} log --oneline -1

In [ ]:
# Script detection used for data filtering and for the "did it answer in the right language?" diagnostic
SCRIPT_RANGES = {
    "Devanagari": (0x0900, 0x097F), "Bengali": (0x0980, 0x09FF), "Gujarati": (0x0A80, 0x0AFF),
    "Odia": (0x0B00, 0x0B7F), "Tamil": (0x0B80, 0x0BFF), "Telugu": (0x0C00, 0x0C7F),
    "Kannada": (0x0C80, 0x0CFF), "Malayalam": (0x0D00, 0x0D7F),
}
LANG_SCRIPT = {
    "Hindi": "Devanagari", "Marathi": "Devanagari", "Sanskrit": "Devanagari",
    "Bengali": "Bengali", "Assamese": "Bengali", "Gujarati": "Gujarati", "Odia": "Odia",
    "Tamil": "Tamil", "Telugu": "Telugu", "Kannada": "Kannada", "Malayalam": "Malayalam", "English": "Latin",
}
TARGET_SCRIPT = LANG_SCRIPT[CFG["eval_language"]]

def script_frac(text, script=TARGET_SCRIPT):
    letters = [c for c in text if c.isalpha()]
    if not letters:
        return 0.0
    if script == "Latin":
        return sum(c.isascii() for c in letters) / len(letters)
    lo, hi = SCRIPT_RANGES[script]
    return sum(lo <= ord(c) <= hi for c in letters) / len(letters)

## 2. Load the model (4-bit NF4 = QLoRA)

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16,
)

def load_base_model():
    kw = dict(device_map={"": 0}, dtype=torch.bfloat16)
    if CFG["use_4bit"]:
        kw["quantization_config"] = bnb_config
    # AutoModelForImageTextToText resolves to Qwen2VL / Qwen2_5_VL ...ForConditionalGeneration
    return AutoModelForImageTextToText.from_pretrained(CFG["model_id"], **kw)

processor = AutoProcessor.from_pretrained(CFG["model_id"], min_pixels=CFG["min_pixels"], max_pixels=CFG["max_pixels"])
gen_processor = AutoProcessor.from_pretrained(CFG["model_id"], min_pixels=CFG["min_pixels"], max_pixels=CFG["eval_max_pixels"])
tokenizer = processor.tokenizer
tokenizer.padding_side = "right"

model = load_base_model()
model.config.use_cache = False
if CFG["use_4bit"]:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,
                                            gradient_checkpointing_kwargs={"use_reentrant": False})
else:
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.enable_input_require_grads()

lora_config = LoraConfig(
    r=CFG["lora_r"], lora_alpha=CFG["lora_alpha"], lora_dropout=CFG["lora_dropout"],
    bias="none", task_type="CAUSAL_LM",
    # language-model projections only. The regex excludes the vision tower, which matters for
    # Qwen2.5-VL, whose vision MLP also has gate/up/down_proj.
    target_modules=r"^(?!.*visual).*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print(f"GPU memory after load: {torch.cuda.memory_allocated()/1e9:.1f} GB")

## 3. Generation helper (batched; can switch the adapter off so the base model is scored with the same code)

In [ ]:
from contextlib import nullcontext
from tqdm.auto import tqdm

def generate(images, prompts, max_new_tokens=256, use_adapter=True, bs=None, desc="generating"):
    bs = bs or CFG["gen_bs"]
    model.eval()
    tok = gen_processor.tokenizer
    tok.padding_side = "left"                     # left-pad for batched decoding
    outs = []
    ctx = nullcontext() if use_adapter else model.disable_adapter()
    with ctx, torch.inference_mode():
        for i in tqdm(range(0, len(prompts), bs), desc=desc):
            msgs = [[{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": p}]}]
                    for p in prompts[i:i + bs]]
            texts = [gen_processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in msgs]
            inputs = gen_processor(text=texts, images=images[i:i + bs], return_tensors="pt", padding=True).to(model.device)
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                 repetition_penalty=1.05, use_cache=True)
            gen = out[:, inputs["input_ids"].shape[1]:]
            for g in gen:
                n_tok = int((g != tok.pad_token_id).sum())
                outs.append({"text": tok.decode(g, skip_special_tokens=True).strip(),
                             "truncated": n_tok >= max_new_tokens})
    tok.padding_side = "right"
    return outs

## 4. Chitrakshara-Cap → clean, in-language caption pairs

Chitrakshara stores image **URLs**. Many are dead or placeholders, and captions span many Indic languages. We filter by script before downloading, clean captions, download in parallel and keep `n_train + n_eval` good pairs in RAM.

In [ ]:
from datasets import load_dataset
import requests
from io import BytesIO
from PIL import Image
from concurrent.futures import ThreadPoolExecutor, as_completed

raw = load_dataset(CFG["dataset_id"], CFG["dataset_config"], split="train")
print(raw)
raw = raw.shuffle(seed=CFG["seed"]).select(range(min(CFG["candidate_pool"], len(raw))))

HEADERS = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/120 Safari/537.36"}
BAD_URL_TOKENS = ("lazy", "placeholder", "default", "blank", "spacer", "logo", "loader", "pixel", "1x1", "noimage", "icon", "avatar")
PREFIX_RE = re.compile(r"^\s*(photo|image|picture|pic|file photo|फोटो|तस्वीर|चित्र)\s*(of|:|-)?\s*[:\-–]?\s*", re.I)
URL_RE = re.compile(r"https?://\S+|www\.\S+")

def clean_caption(t):
    t = URL_RE.sub("", (t or "")).strip()
    t = PREFIX_RE.sub("", t).strip(" |-–:")
    t = re.sub(r"\s+", " ", t)
    return t

def keep_row(url, text):
    if not isinstance(url, str) or not url.startswith("http") or any(b in url.lower() for b in BAD_URL_TOKENS):
        return False
    if not (CFG["min_caption_chr"] <= len(text) <= CFG["max_caption_chr"]):
        return False
    if CFG["filter_script"] and script_frac(text) < CFG["min_script_frac"]:
        return False
    return True

def fetch_image(url):
    try:
        r = requests.get(url, timeout=6, headers=HEADERS)
        if r.status_code != 200 or len(r.content) < 4000:
            return None
        img = Image.open(BytesIO(r.content))
        img = img.convert("RGBA").convert("RGB") if img.mode in ("P", "LA", "RGBA") else img.convert("RGB")
        w, h = img.size
        if min(w, h) < 64 or max(w, h) / min(w, h) > 8:
            return None
        img.thumbnail((CFG["max_image_side"], CFG["max_image_side"]))
        return img
    except Exception:
        return None

candidates, seen_urls, seen_caps = [], set(), set()
for row in raw:
    url, text = row["image"], clean_caption(row["text"])
    if url in seen_urls or text in seen_caps or not keep_row(url, text):
        continue
    seen_urls.add(url); seen_caps.add(text)
    candidates.append((url, text))
print(f"{len(candidates)} candidate rows after URL/caption/script filtering (script={TARGET_SCRIPT})")

target = CFG["n_train"] + CFG["n_eval"]
pairs = []
with ThreadPoolExecutor(max_workers=64) as ex:
    futs = {ex.submit(fetch_image, u): t for u, t in candidates}
    for f in tqdm(as_completed(futs), total=len(futs), desc="downloading"):
        img = f.result()
        if img is not None:
            pairs.append((img, futs[f]))
        if len(pairs) >= target:
            for g in futs: g.cancel()
            break

print(f"Usable image-caption pairs: {len(pairs)}")
assert len(pairs) >= 500, "Too few images - raise candidate_pool, or set filter_script=False."
random.shuffle(pairs)
n_eval = min(CFG["n_eval"], max(20, len(pairs) // 25))
eval_pairs, train_pairs = pairs[:n_eval], pairs[n_eval:target]
print(f"train={len(train_pairs)}  eval={len(eval_pairs)}")

## 5. Build the training mix

* **Caption task**: the caption is the target. The prompt explicitly asks for a *short caption*, so the model does not learn to answer "describe in detail" with a headline. Several prompt phrasings are used, half of them in the target language.
* **Self-distillation (anti-forgetting)**: for `n_distill` training images, the *base* model writes a detailed answer in the target language. Only answers that are mostly in the right script and not truncated or repetitive are kept. Mixing these in keeps (and nudges into Hindi) the long-form behaviour that LLaVA-Bench rewards.

In [ ]:
CAPTION_PROMPTS_EN = [
    "Write a short one-line caption for this image.",
    "Give a brief headline-style caption for this picture.",
    "Write a concise alt-text caption for this image.",
]
CAPTION_PROMPTS_HI = [
    "इस चित्र के लिए एक छोटा शीर्षक लिखें।",
    "इस तस्वीर का एक पंक्ति में कैप्शन दें।",
    "इस छवि के लिए एक संक्षिप्त कैप्शन लिखिए।",
]
DETAIL_PROMPTS = {
    "Hindi": ["इस चित्र में क्या-क्या दिखाई दे रहा है? हिंदी में विस्तार से बताइए।",
              "इस छवि का हिंदी में विस्तृत विवरण दीजिए।"],
}
DETAIL_PROMPTS_EN = [f"Describe everything in this image in detail. Answer in {CFG['eval_language']}."]
detail_prompts = DETAIL_PROMPTS.get(CFG["eval_language"], DETAIL_PROMPTS_EN)
caption_prompts = CAPTION_PROMPTS_EN + (CAPTION_PROMPTS_HI if TARGET_SCRIPT == "Devanagari" else [])

def caption_example(img, cap):
    return {"image": img, "prompt": random.choice(caption_prompts), "target": cap, "max_tok": CFG["max_caption_tok"]}

def is_repetitive(t, n=4):
    w = t.split()
    grams = [tuple(w[i:i + n]) for i in range(max(0, len(w) - n + 1))]
    return len(grams) > 10 and len(set(grams)) / len(grams) < 0.6

train_examples = [caption_example(i, c) for i, c in train_pairs]
eval_examples  = [caption_example(i, c) for i, c in eval_pairs]

if CFG["n_distill"]:
    d_imgs = [img for img, _ in train_pairs[: CFG["n_distill"]]]
    d_prompts = [random.choice(detail_prompts) for _ in d_imgs]
    d_out = generate(d_imgs, d_prompts, max_new_tokens=CFG["distill_max_tok"], use_adapter=False, desc="self-distill")
    kept = [(im, p, o["text"]) for im, p, o in zip(d_imgs, d_prompts, d_out)
            if not o["truncated"] and len(o["text"]) > 80 and script_frac(o["text"]) >= 0.6 and not is_repetitive(o["text"])]
    print(f"self-distillation: kept {len(kept)}/{len(d_imgs)} detailed answers")
    if kept:
        print("example:", kept[0][2][:300])
    train_examples += [{"image": im, "prompt": p, "target": t, "max_tok": CFG["distill_max_tok"]} for im, p, t in kept]

random.shuffle(train_examples)
print(f"training examples: {len(train_examples)}")

## 6. Dataset + collator (loss only on the assistant tokens)

In [ ]:
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence

ASSISTANT_TAG = tokenizer("<|im_start|>assistant\n", add_special_tokens=False)["input_ids"]

def find_last(seq, sub):
    for i in range(len(seq) - len(sub), -1, -1):
        if seq[i:i + len(sub)] == sub:
            return i
    return -1

class SFTDataset(Dataset):
    def __init__(self, examples): self.examples = examples
    def __len__(self): return len(self.examples)
    def __getitem__(self, idx):
        ex = self.examples[idx]
        msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": ex["prompt"]}]}]
        prompt = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        ids = tokenizer(ex["target"], add_special_tokens=False)["input_ids"][: ex["max_tok"]]
        enc = processor(text=[prompt + tokenizer.decode(ids) + "<|im_end|>"], images=[ex["image"]], return_tensors="pt")
        input_ids = enc["input_ids"][0]
        labels = input_ids.clone()
        start = find_last(input_ids.tolist(), ASSISTANT_TAG)
        assert start >= 0
        labels[: start + len(ASSISTANT_TAG)] = -100
        item = {"input_ids": input_ids, "attention_mask": enc["attention_mask"][0], "labels": labels,
                "pixel_values": enc["pixel_values"], "image_grid_thw": enc["image_grid_thw"].view(-1, 3)}
        if "mm_token_type_ids" in enc:
            item["mm_token_type_ids"] = enc["mm_token_type_ids"][0]
        return item

def collate_fn(examples):
    pad_id = tokenizer.pad_token_id
    batch = {
        "input_ids":      pad_sequence([e["input_ids"] for e in examples], batch_first=True, padding_value=pad_id),
        "attention_mask": pad_sequence([e["attention_mask"] for e in examples], batch_first=True, padding_value=0),
        "labels":         pad_sequence([e["labels"] for e in examples], batch_first=True, padding_value=-100),
        "pixel_values":   torch.cat([e["pixel_values"] for e in examples], dim=0),
        "image_grid_thw": torch.cat([e["image_grid_thw"] for e in examples], dim=0),
    }
    if "mm_token_type_ids" in examples[0]:
        batch["mm_token_type_ids"] = pad_sequence([e["mm_token_type_ids"] for e in examples], batch_first=True, padding_value=0)
    return batch

train_dataset, eval_dataset = SFTDataset(train_examples), SFTDataset(eval_examples)
s = train_dataset[0]
print({k: tuple(v.shape) for k, v in s.items()})
print("supervised text:", tokenizer.decode(s["labels"][s["labels"] != -100]))

## 7. Train (early stopping on eval loss, best checkpoint restored)

In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir=OUT_DIR,
    per_device_train_batch_size=CFG["per_device_bs"], per_device_eval_batch_size=CFG["per_device_bs"],
    gradient_accumulation_steps=CFG["grad_accum"],
    num_train_epochs=CFG["epochs"], learning_rate=CFG["lr"], lr_scheduler_type=CFG["lr_scheduler"],
    warmup_ratio=CFG["warmup_ratio"], weight_decay=CFG["weight_decay"], max_grad_norm=CFG["max_grad_norm"],
    optim="paged_adamw_8bit", bf16=True, fp16=False,
    gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=10,
    eval_strategy="steps", eval_steps=CFG["eval_steps"],
    save_strategy="steps", save_steps=CFG["eval_steps"], save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
    remove_unused_columns=False, label_names=["labels"],
    dataloader_num_workers=2, report_to="none", seed=CFG["seed"],
)
trainer = Trainer(
    model=model, args=training_args,
    train_dataset=train_dataset, eval_dataset=eval_dataset, data_collator=collate_fn,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=CFG["early_stop_patience"])],
)
torch.cuda.empty_cache()
train_result = trainer.train()
print(train_result.metrics)
print(f"Peak GPU memory: {torch.cuda.max_memory_allocated()/1e9:.1f} GB")

In [ ]:
import matplotlib.pyplot as plt
log = trainer.state.log_history
tr = [(l["step"], l["loss"]) for l in log if "loss" in l]
ev = [(l["step"], l["eval_loss"]) for l in log if "eval_loss" in l]
plt.figure(figsize=(8, 4))
plt.plot(*zip(*tr), label="train loss")
if ev: plt.plot(*zip(*ev), "o-", label="eval loss (held-out captions)")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()
if ev: print(f"best eval loss: {min(v for _, v in ev):.3f}  (checkpoint: {trainer.state.best_model_checkpoint})")

## 8. Save the adapter (and reload it if you start a fresh runtime here)

In [ ]:
model = trainer.model
model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
with open(f"{SAVE_DIR}/run_config.json", "w") as f:
    json.dump(CFG, f, indent=2)
print("saved to", SAVE_DIR)
# Optional: copy to Drive so it survives the runtime
# from google.colab import drive; drive.mount("/content/drive"); !cp -r {SAVE_DIR} /content/drive/MyDrive/

In [ ]:
# Only needed for an evaluation-only session (cells 1-3 first, skip training):
if "model" not in globals() or not isinstance(model, PeftModel):
    model = PeftModel.from_pretrained(load_base_model(), SAVE_DIR)
model.eval()
model.config.use_cache = True
torch.cuda.empty_cache()

img, ref = eval_pairs[1] if "eval_pairs" in globals() else (None, None)
if img is not None:
    p = random.choice(caption_prompts)
    display(img)
    print("reference :", ref)
    print("base      :", generate([img], [p], 64, use_adapter=False)[0]["text"])
    print("finetuned :", generate([img], [p], 64, use_adapter=True)[0]["text"])

## 9. Indic-LLaVA-Bench: base vs. fine-tuned

The 60 questions and ids come from the repo (`Indic-LLaVABench-Evaluator/inputs/<Lang>/questions.jsonl`). The judge compares each answer against the GPT-4 reference answer and reports a **relative score** (model / GPT-4 × 100), per category (`conv`, `detail`, `complex`).

In [ ]:
LANG = CFG["eval_language"]
LB_DIR = f"{BB_DIR}/Indic-LLaVABench-Evaluator"
questions = [json.loads(l) for l in open(f"{LB_DIR}/inputs/{LANG}/questions.jsonl", encoding="utf-8")]
print(len(questions), "questions;", len({q['image'] for q in questions}), "images")

def load_llava_bench_images(needed):
    # 1) the image folder referenced by the repo ReadMe
    try:
        from huggingface_hub import snapshot_download
        root = snapshot_download("MBZUAI/multilingual-llava-bench-in-the-wild", repo_type="dataset",
                                 allow_patterns=["images/*"])
        imgs = {n: Image.open(os.path.join(root, "images", n)).convert("RGB") for n in needed
                if os.path.exists(os.path.join(root, "images", n))}
        if len(imgs) == len(needed):
            return imgs
        print(f"MBZUAI repo had {len(imgs)}/{len(needed)} images, trying fallback")
    except Exception as e:
        print("MBZUAI download failed:", e)
    # 2) lmms-lab copy of LLaVA-Bench-in-the-wild (image_id like "001")
    ds = load_dataset("lmms-lab/llava-bench-in-the-wild", split="train")
    imgs = {}
    for r in ds:
        name = f"{str(r['image_id']).split('.')[0]}.jpg"
        if name in needed and name not in imgs:
            imgs[name] = r["image"].convert("RGB")
    missing = set(needed) - set(imgs)
    assert not missing, f"missing images: {sorted(missing)}"
    return imgs

LB_IMAGES = load_llava_bench_images(sorted({q["image"] for q in questions}))

LANG_HINT = {"Hindi": "कृपया हिंदी में उत्तर दें।"}
def lb_prompt(q):
    if CFG["answer_lang_hint"]:
        return q["text"] + "\n" + LANG_HINT.get(LANG, f"Please answer in {LANG}.")
    return q["text"]

def run_llava_bench(tag, use_adapter):
    outs = generate([LB_IMAGES[q["image"]] for q in questions], [lb_prompt(q) for q in questions],
                    max_new_tokens=CFG["llava_max_new_tokens"], use_adapter=use_adapter, desc=f"LLaVA-Bench {tag}")
    path = f"{LB_DIR}/model_responses/responses_{LANG}_{tag}.jsonl"
    with open(path, "w", encoding="utf-8") as f:
        for q, o in zip(questions, outs):          # same order and ids as questions.jsonl
            f.write(json.dumps({"image": q["image"], "text": q["text"], "category": q["category"],
                                "question_id": q["question_id"], "response": o["text"],
                                "truncated": o["truncated"], "model_name": tag}, ensure_ascii=False) + "\n")
    print("wrote", path)
    return outs

TAG_BASE, TAG_FT = "qwen2vl_base", CFG["run_name"]
lb_outputs = {TAG_BASE: run_llava_bench(TAG_BASE, use_adapter=False),
              TAG_FT:   run_llava_bench(TAG_FT,   use_adapter=True)}
for t, o in lb_outputs.items():
    print(f"\n[{t}] Q: {questions[1]['text']}\n{o[1]['text'][:400]}")

In [ ]:
# GPT judge. Pick the judge used for the numbers you compare against: gpt-4-0613 is the repo default.
# If your key cannot use it, set e.g. "gpt-4o", but then scores are NOT comparable to published numbers.
JUDGE_MODEL = os.environ.get("JUDGE_MODEL", "gpt-4-0613")

import subprocess
def judge_llava_bench(tag):
    resp = f"./model_responses/responses_{LANG}_{tag}.jsonl"
    ans  = f"./answers/answer_{LANG}_{tag}.jsonl"
    rev  = f"./reviews/review_{LANG}_{tag}_{JUDGE_MODEL}.jsonl"
    if not os.path.exists(os.path.join(LB_DIR, resp)):
        print("no responses for", tag); return None
    cmds = [
        ["python", "scripts/answer_file.py", "--response-file", resp, "--answers-file", ans, "--overwrite"],
        ["python", "scripts/eval_gpt_review_bench.py",
         "--question", f"./inputs/{LANG}/questions.jsonl", "--context", f"./inputs/{LANG}/context.jsonl",
         "--rule", "./scripts/table/rule.json", "--answer-list", f"./inputs/{LANG}/answers_gpt4.jsonl", ans,
         "--output", rev, "--judge-model", JUDGE_MODEL],
    ]
    # reviews are resumable (the script skips rows already in the file); delete `rev` to re-judge from scratch
    for c in cmds:
        r = subprocess.run(c, cwd=LB_DIR, capture_output=True, text=True)
        if r.returncode != 0:
            print(r.stdout[-2000:], r.stderr[-3000:]); raise RuntimeError(f"failed: {' '.join(c)}")
    return os.path.join(LB_DIR, rev)

def llava_scores(review_path):
    from collections import defaultdict
    sc = defaultdict(list)
    for l in open(review_path, encoding="utf-8"):
        r = json.loads(l)
        if r["tuple"][0] < 0:              # judge output could not be parsed
            continue
        cat = r["category"].replace("llava_bench_", "")
        sc[cat].append(r["tuple"]); sc["all"].append(r["tuple"])
    res = {}
    for k, v in sc.items():
        gpt4, mdl = np.asarray(v).mean(0)
        res[k] = round(mdl / gpt4 * 100, 1)        # relative score, same formula as summarize_gpt_review.py
        if k == "all": res["abs_all"] = round(mdl * 10, 1)
    return res

LB_RESULTS = {}
# Add "chitrarth" here if you put Chitrarth's responses in model_responses/responses_<Lang>_chitrarth.jsonl
# (same format as the files written above, produced with Chitrarth's own inference code).
LB_TAGS = [TAG_BASE, TAG_FT] + (["chitrarth"] if os.path.exists(f"{LB_DIR}/model_responses/responses_{LANG}_chitrarth.jsonl") else [])
if os.environ.get("OPENAI_API_KEY"):
    for t in LB_TAGS:
        rev = judge_llava_bench(t)
        if rev: LB_RESULTS[t] = llava_scores(rev)
    print(json.dumps(LB_RESULTS, indent=2))
else:
    print("OPENAI_API_KEY not set: skipping the LLaVA-Bench judge (responses are saved, judge later).")

## 10. Indic-POPE (object hallucination, yes/no, no API key needed)

Questions come from `Indic-POPE-Evaluator/question_files/pope_test_<lang>.jsonl`, labels from `inputs/coco/coco_pope_<category>.json` (aligned by position within each category, as in the repo scorer), and COCO images from `lmms-lab/POPE`. Answers are mapped to yes/no with the repo's `language_statements.json`. With `pope_n_per_cat=None` the full 8,910-question set is used, which is what published numbers use.

In [ ]:
PP_DIR = f"{BB_DIR}/Indic-POPE-Evaluator"
lang_l = LANG.lower()
pope_q = [json.loads(l) for l in open(f"{PP_DIR}/question_files/pope_test_{lang_l}.jsonl", encoding="utf-8")]
STATEMENTS = json.load(open(f"{PP_DIR}/inputs/language_statements.json", encoding="utf-8"))

pope_items = []
for cat in ["adversarial", "popular", "random"]:
    qs = [q for q in pope_q if q["category"] == cat]
    labels = [json.loads(l) for l in open(f"{PP_DIR}/inputs/coco/coco_pope_{cat}.json", encoding="utf-8")]
    assert len(qs) == len(labels)
    items = [dict(q, label=lab["label"]) for q, lab in zip(qs, labels) if q["image"] == lab["image"]]
    if CFG["pope_n_per_cat"]:
        items = random.Random(CFG["seed"]).sample(items, min(CFG["pope_n_per_cat"], len(items)))
    pope_items += items
print(f"POPE questions: {len(pope_items)}")

pope_ds = load_dataset("lmms-lab/POPE", split="test")
src_to_idx = {}
for i, s in enumerate(pope_ds["image_source"]):
    src_to_idx.setdefault(str(s).replace(".jpg", ""), i)
need = sorted({q["image"].replace(".jpg", "") for q in pope_items})
missing = [n for n in need if n not in src_to_idx]
assert not missing, f"{len(missing)} COCO images not found in lmms-lab/POPE, e.g. {missing[:3]}"
POPE_IMAGES = {n: pope_ds[src_to_idx[n]]["image"].convert("RGB") for n in tqdm(need, desc="POPE images")}

def _norm(t):
    # unify chandrabindu/anusvara spellings (हाँ vs हां) and strip punctuation-like whitespace
    return t.strip().lower().replace("\u0901", "\u0902")

def to_yes_no(ans, lang=lang_l):
    a = _norm(ans)
    st = STATEMENTS.get(lang, STATEMENTS["english"])
    neg = sorted({_norm(p) for p in st["negative"] + STATEMENTS["english"]["negative"]}, key=len, reverse=True)
    aff = sorted({_norm(p) for p in st["affirmative"] + STATEMENTS["english"]["affirmative"]}, key=len, reverse=True)
    def starts(phrases): return any(a.startswith(p) for p in phrases)
    def contains(phrases):
        return any((re.search(rf"\b{re.escape(p)}\b", a) if p.isascii() else p in a) for p in phrases)
    if starts(neg): return "no"          # longest phrases first, so "सही नहीं" wins over "सही"
    if starts(aff): return "yes"
    if contains(neg): return "no"
    if contains(aff): return "yes"
    return "no"                          # POPE convention: anything that is not a yes counts as no

def pope_metrics(preds, items):
    out = {}
    for cat in ["adversarial", "popular", "random", "all"]:
        sel = [(p, it["label"]) for p, it in zip(preds, items) if cat == "all" or it["category"] == cat]
        tp = sum(p == "yes" and l == "yes" for p, l in sel); fp = sum(p == "yes" and l == "no" for p, l in sel)
        tn = sum(p == "no" and l == "no" for p, l in sel);  fn = sum(p == "no" and l == "yes" for p, l in sel)
        prec = tp / (tp + fp) if tp + fp else 0; rec = tp / (tp + fn) if tp + fn else 0
        out[cat] = dict(acc=round(100 * (tp + tn) / len(sel), 1), f1=round(100 * (2 * prec * rec / (prec + rec) if prec + rec else 0), 1),
                        yes_ratio=round(100 * (tp + fp) / len(sel), 1))
    return out

POPE_RESULTS, pope_raw = {}, {}
for tag, use_ad in [(TAG_BASE, False), (TAG_FT, True)]:
    outs = generate([POPE_IMAGES[q["image"].replace(".jpg", "")] for q in pope_items], [q["text"] for q in pope_items],
                    max_new_tokens=16, use_adapter=use_ad, bs=16, desc=f"POPE {tag}")
    pope_raw[tag] = [o["text"] for o in outs]
    POPE_RESULTS[tag] = pope_metrics([to_yes_no(t) for t in pope_raw[tag]], pope_items)
    with open(f"{PP_DIR}/model_responses/{lang_l}_results_{tag}.jsonl", "w", encoding="utf-8") as f:
        for q, t in zip(pope_items, pope_raw[tag]):
            f.write(json.dumps({"question_id": q["question_id"], "prompt": q["text"], "text": t,
                                "category": q["category"], "label": q["label"], "model_id": tag}, ensure_ascii=False) + "\n")
print(json.dumps(POPE_RESULTS, indent=2))
print("sample raw answers:", pope_raw[TAG_FT][:8])

## 11. Comparison with Chitrarth

Chitrarth's BharatBench numbers are reported in the paper (*Chitrarth: Bridging Vision and Language for a Billion People*, arXiv:2502.15392) and on the `krutrim-ai-labs/Chitrarth` model card. **Copy the values for your language into `CHITRARTH_REF`** (they are left blank instead of guessed). Before comparing, check that the paper reports the same quantity (LLaVA-Bench relative vs. absolute score, POPE accuracy vs. F1), the same judge model and the full question set. A different judge or a POPE subset makes the comparison indicative only.

For a strictly fair comparison, run Chitrarth's own inference on the same questions, save `responses_<Lang>_chitrarth.jsonl` (LLaVA-Bench) next to the files above, and re-run the judge cell. It is then scored by the identical pipeline.

In [ ]:
import pandas as pd

CHITRARTH_REF = {   # fill in from the paper / model card for CFG["eval_language"]
    "llava_bench": None,   # LLaVA-Bench score
    "pope": None,          # POPE score (accuracy or F1, as reported)
    "mmvet": None,         # MM-Vet score (not computed in this notebook)
}

rows = []
for tag, name in [(TAG_BASE, "Qwen2-VL-7B-Instruct (base, 4-bit)" if CFG["use_4bit"] else "Qwen2-VL-7B-Instruct (base)"),
                  (TAG_FT, "Qwen2-VL-7B + Chitrakshara (Q)LoRA")] + ([("chitrarth", "Chitrarth (same pipeline)")] if "chitrarth" in LB_RESULTS else []):
    lb = LB_RESULTS.get(tag, {}); pp = POPE_RESULTS.get(tag, {}).get("all", {})
    rows.append({"model": name, "LLaVA-Bench rel. (all)": lb.get("all"), "conv": lb.get("conv"),
                 "detail": lb.get("detail"), "complex": lb.get("complex"),
                 "POPE acc": pp.get("acc"), "POPE F1": pp.get("f1"), "POPE yes%": pp.get("yes_ratio")})
rows.append({"model": "Chitrarth (paper)", "LLaVA-Bench rel. (all)": CHITRARTH_REF["llava_bench"],
             "POPE acc": CHITRARTH_REF["pope"]})
table = pd.DataFrame(rows).set_index("model")
print(f"Language: {LANG} | judge: {JUDGE_MODEL} | POPE questions: {len(pope_items)}"
      + ("  (subset -> indicative only)" if CFG["pope_n_per_cat"] else ""))
display(table)
table.to_csv(f"{OUT_DIR}/bharatbench_{LANG}_comparison.csv")

if CHITRARTH_REF["llava_bench"] and LB_RESULTS.get(TAG_FT):
    gap = CHITRARTH_REF["llava_bench"] - LB_RESULTS[TAG_FT]["all"]
    print(f"LLaVA-Bench gap to Chitrarth: {gap:+.1f} points")

## 12. Diagnostics: why the scores are what they are

In [ ]:
def diag(outs):
    texts = [o["text"] for o in outs]
    return {
        "mean chars": round(np.mean([len(t) for t in texts])),
        f"% in {TARGET_SCRIPT} (>=60% of letters)": round(100 * np.mean([script_frac(t) >= 0.6 for t in texts]), 1),
        "% truncated": round(100 * np.mean([o["truncated"] for o in outs]), 1),
        "% repetitive": round(100 * np.mean([is_repetitive(t) for t in texts]), 1),
    }
d = pd.DataFrame({t: diag(o) for t, o in lb_outputs.items()}).T
for t in lb_outputs:
    for cat in ["conv", "detail", "complex"]:
        d.loc[t, f"chars[{cat}]"] = round(np.mean([len(o["text"]) for o, q in zip(lb_outputs[t], questions) if q["category"] == cat]))
display(d)

base, ft = lb_outputs[TAG_BASE], lb_outputs[TAG_FT]
tips = []
if d.loc[TAG_FT, "mean chars"] < 0.6 * d.loc[TAG_BASE, "mean chars"]:
    tips.append("Fine-tuned answers are much shorter than the base model's: the caption task is overriding long-form answering. "
                "Raise n_distill, lower lr/epochs, or add real instruction data (see notes).")
if d.iloc[:, 1].min() < 80:
    tips.append(f"Many answers are not in {LANG}. The judge penalises off-language answers: add in-language instruction data "
                "or report an extra run with answer_lang_hint=True.")
if d.loc[TAG_FT, "% repetitive"] > 5 or d.loc[TAG_FT, "% truncated"] > 10:
    tips.append("Repetition/truncation in Indic generations: raise repetition_penalty to 1.1 or add no_repeat_ngram_size=6.")
if POPE_RESULTS:
    yr = POPE_RESULTS[TAG_FT]["all"]["yes_ratio"]
    if yr > 60: tips.append(f"POPE yes-ratio {yr}%: the model over-says 'yes' (hallucination). Add negative/absent-object QA to training.")
    if POPE_RESULTS[TAG_FT]["all"]["acc"] < POPE_RESULTS[TAG_BASE]["all"]["acc"] - 1:
        tips.append("POPE dropped after fine-tuning: captions do not teach grounding. Mix in yes/no grounding data or freeze more (lower r).")
if LB_RESULTS.get(TAG_FT) and LB_RESULTS.get(TAG_BASE):
    for cat in ["conv", "detail", "complex"]:
        delta = LB_RESULTS[TAG_FT].get(cat, 0) - LB_RESULTS[TAG_BASE].get(cat, 0)
        print(f"LLaVA-Bench {cat:8s}: {delta:+.1f} vs base")
print("\n".join("- " + t for t in tips) or "No obvious failure pattern; see the notes below for next steps.")

## 13. How to close the gap to Chitrarth (in rough order of expected impact)

1. **Fix the task mismatch.** Chitrakshara-Cap is web alt-text (short, often news headlines). BharatBench measures open-ended VQA (LLaVA-Bench) and grounding (POPE), so caption-only fine-tuning mostly teaches *language and script* and can hurt instruction following. Train a second stage on **Indic multimodal instruction data** (e.g. LLaVA-Instruct-style conversations translated into Hindi/Indic languages with IndicTrans2) and on Chitrakshara-IL (interleaved). Instruction data in the target languages is what the benchmark rewards.
2. **Answer in the right language.** Check the `% in script` column. Qwen2-VL often answers Hindi questions in English or in mixed language, and the judge penalises that. In-language instruction data fixes it properly; `answer_lang_hint=True` is a prompt-level fix (report it as a separate setting).
3. **Stronger or less degraded base.** Try `Qwen/Qwen2.5-VL-7B-Instruct`, and on an 80 GB A100 `use_4bit=False` (bf16 LoRA). NF4 quantisation costs some quality for both the baseline and the fine-tuned model.
4. **Data quality over quantity.** Filter Chitrakshara pairs by image-text similarity (e.g. a multilingual SigLIP/CLIP score), drop generic headlines and boilerplate, dedupe near-identical captions, then scale `n_train` (10k–50k) once the eval loss and the benchmark deltas are positive.
5. **Resolution.** Raise `max_pixels` (train) and `eval_max_pixels` for detail/OCR-heavy questions. This costs memory and time.
6. **Hyper-parameters.** With mixed data, `lora_r` 16–64, `lr` 5e-5–2e-4, 1–2 epochs. Pick by **benchmark delta vs. base**, not by train loss.
7. **Evaluate like the paper.** Use the same judge (`gpt-4-0613` by default), all 60 LLaVA-Bench questions, full POPE (`pope_n_per_cat=None`), and several languages (`eval_language`), then report the per-language average.